**Libraries Installation**

In [1]:
## Install necessary libraries here ##
%pip install neo4j
%pip install neo4j-driver
%pip install neo4j-graphrag
%pip install tiktoken
%pip install ollama
%pip install networkx
%pip install tqdm
%pip install requests

## Import libraries here ##
import neo4j
import neo4j_graphrag
import requests
import ollama
import re
import json
from typing import List, Dict
from tqdm import tqdm
import json

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


**NEO4J Connection Initiation**

In [ ]:
from neo4j import GraphDatabase
neo4j_driver = GraphDatabase.driver(
    "neo4j+s://7bd18831.databases.neo4j.io",
    auth=("7bd18831", "VfZDa9mphM-v6KV0CduR-VKchaD2LDxJHi_YZzvMCxc"),
    notifications_min_severity="OFF")
try:
    neo4j_driver.verify_connectivity()
    print("Connection successful!")
except Exception as e:
    print(f"Connection failed: {e}")

# Add this cell before any import of data
# neo4j_driver.execute_query("MATCH (n) DETACH DELETE n")
# print("Database cleared.")

**API KEY LOAD**

In [ ]:
import os
import requests
from pathlib import Path

# =============================================================================
# LOCAL LLAMA ENVIRONMENT CONFIGURATION (QWEN)
# =============================================================================

# Default local server URLs:
# - Ollama: "http://localhost:11434/v1"
# - Llama.cpp / vLLM / LM Studio: "http://localhost:8000/v1"
LOCAL_API_BASE = "https://crewless-snagged-facecloth.ngrok-free.dev/v1"

# Specify the exact tag of the Qwen model you pulled locally (e.g., "qwen2.5", "qwen2.5:7b")
LOCAL_MODEL_NAME = "qwen2.5:7b"

print(f"Local Llama Environment Initialized.")
print(f"   - Target Base URL: {LOCAL_API_BASE}")
print(f"   - Target Model:    {LOCAL_MODEL_NAME}")

def verify_local_connection():
    """Verify that your local Llama/Ollama server is active and responding."""
    try:
        # Check basic models endpoint
        response = requests.get(f"{LOCAL_API_BASE}/models", timeout=50)
        if response.status_code == 200:
            print("Connection successful! Local server is online.")
            available_models = [m.get("id") for m in response.json().get("data", [])]
            print(f"Available local models: {available_models}")
        else:
            print(f"Server responded with status code: {response.status_code}")
    except Exception as e:
        print(f"Connection failed: Is your local server running at {LOCAL_API_BASE}? Error: {e}")

verify_local_connection()

**Embedded and Chat Extraction Logic**

In [ ]:
import time
import re
import requests

# Global fallback configuration variables (if not already defined in your notebook)
LOCAL_API_BASE = globals().get("LOCAL_API_BASE", "https://crewless-snagged-facecloth.ngrok-free.dev/v1")
LOCAL_MODEL_NAME = globals().get("LOCAL_MODEL_NAME", "qwen2.5:7b")
LOCAL_EMBED_MODEL = globals().get("LOCAL_EMBED_MODEL", "qwen3-embedding:latest")

# ---------- Local Embeddings ----------
def embed(texts, model=LOCAL_EMBED_MODEL, max_retries=3):
    """
    Generate embeddings using your local server.
    Handles both a single string or a list of text strings.
    """
    url = f"{LOCAL_API_BASE}/embeddings"
    payload = {
        "model": model,
        "input": texts
    }
    
    for attempt in range(max_retries):
        try:
            response = requests.post(url, json=payload, timeout=60)
            response.raise_for_status()
            response_data = response.json()
            
            # Local endpoints can return variations depending on if input was a string or list
            data_records = response_data.get("data", [])
            if isinstance(texts, list):
                return [record["embedding"] for record in data_records]
            else:
                return data_records[0]["embedding"]
                
        except Exception as e:
            print(f"⚠️ Local embedding attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2)
            else:
                raise e

# -------------------------------
# 2. Local Chat Completion (OpenAI-compatible endpoint)
# -------------------------------

def chat(messages, model=LOCAL_MODEL_NAME, temperature=0, config={}, max_retries=3):
    """
    Perform a structured message completion using your local Qwen model.
    No more manual string concatenation—we send the message array directly!
    """
    url = f"{LOCAL_API_BASE}/chat/completions"
    
    # Standard OpenAI Chat Geometry
    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
    }
    payload.update(config)

    for attempt in range(max_retries):
        try:
            response = requests.post(url, json=payload, timeout=120)
            response.raise_for_status()
            response_data = response.json()
            
            # Extract content from choices structure
            content = response_data["choices"][0]["message"]["content"]
            return content.strip()
            
        except Exception as e:
            print(f"⚠️ Local chat completion attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2)
            else:
                raise e

# -------------------------------
# 3. Local Tool‑calling (Function Calling)
# -------------------------------

def tool_choice(messages, model=LOCAL_MODEL_NAME, temperature=0, tools=[], config={}):
    """
    Perform a chat completion that involves local tool selection.
    Uses native OpenAI-style json payloads directly supported by Qwen models.
    """
    url = f"{LOCAL_API_BASE}/chat/completions"
    
    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "tools": tools if tools else None
    }
    payload.update(config)

    try:
        response = requests.post(url, json=payload, timeout=120)
        response.raise_for_status()
        response_json = response.json()
        
        message = response_json["choices"][0]["message"]
        if "tool_calls" in message and message["tool_calls"]:
            return message["tool_calls"]
        return []
    except Exception as e:
        print(f"❌ Local Tool-Call execution failed: {e}")
        return []


# ---------- Token Counting & Text Chunking ----------

def num_tokens_from_string(text: str, model_name: str = "qwen2.5:7b") -> int:
    """
    Counts tokens by querying the local Ollama instance's tokenization endpoint.
    Falls back to a safe approximation if the endpoint is unreachable.
    """
    try:
        url = "https://crewless-snagged-facecloth.ngrok-free.dev/api/tokenize"
        payload = {
            "model": model_name,
            "prompt": text
        }
        response = requests.post(url, json=payload, timeout=15)
        if response.status_code == 200:
            return len(response.json().get("tokens", []))
    except Exception:
        pass
    
    # Fallback estimation for Qwen's tokenizer if Ollama is busy 
    # (Roughly 1 word ≈ 1.3 tokens for English medical/technical text)
    return int(len(text.split()) * 1.3)


def chunk_text(text: str, chunk_size: int = 2000, overlap: int = 400) -> List[str]:
    """
    Splits text into chunks of `chunk_size` characters with a sliding window `overlap`.
    """
    if chunk_size <= overlap:
        raise ValueError("chunk_size must be greater than overlap")
        
    chunks = []
    start = 0
    text_len = len(text)
    
    while start < text_len:
        end = start + chunk_size
        chunks.append(text[start:end].strip())
        
        if end >= text_len:
            break
            
        start += (chunk_size - overlap)
        
    return chunks

print("[✓] Ollama-compatible tokenizing and chunking functions defined.")

**Text2Cypher Function**

In [5]:
import neo4j
from typing import Literal


class Text2Cypher:
    def __init__(self, driver: neo4j.Driver):
        self.driver = driver
        self.dynamic_sections = {}
        self.required_sections = ["question"]
        self.prompt_template = prompt_template

        schema_string = get_schema(driver)
        self.set_prompt_section("schema", schema_string)

    def set_prompt_section(
        self,
        section: Literal["terminology", "examples", "schema", "question"],
        value: str,
    ):
        self.dynamic_sections[section] = value

    def get_full_prompt(self):
        prompt = self.prompt_template["static"]["instructions"]
        # loop through the prompt_template["dynamic"] and add the values from self.dynamic_sections
        for section in self.prompt_template["dynamic"]:
            if section in self.dynamic_sections:
                prompt += self.prompt_template["dynamic"][section].format(
                    self.dynamic_sections[section]
                )
        return prompt

    def generate_cypher(self):
        # check if required sections are set
        for section in self.required_sections:
            if section not in self.dynamic_sections:
                raise ValueError(
                    f"Section {section} is required to generate a prompt. Use set_prompt_section to set it."
                )
        prompt = self.get_full_prompt()
        cypher = chat(messages=[{"role": "user", "content": prompt}])
        return cypher


prompt_template = {
    "static": {
        "instructions": """
    Instructions:
    Generate Cypher statement to query a graph database to get the data to answer the user question below.

    Format instructions:
    Do not include any explanations or apologies in your responses.
    Do not respond to any questions that might ask anything else than for you to
    construct a Cypher statement.
    Do not include any text except the generated Cypher statement.
    ONLY RESPOND WITH CYPHER, NO CODEBLOCKS.
    Make sure to name RETURN variables as requested in the user question.
    """
    },
    "dynamic": {
        "schema": """
    Graph Database Schema:
    Use only the provided relationship types and properties in the schema.
    Do not use any other relationship types or properties that are not provided in the schema.
    {}
    """,
        "terminology": """
    Terminology mapping:
    This section is helpful to map terminology between the user question and the graph database schema.
    {}
    """,
        "examples": """
    Examples:
    The following examples provide useful patterns for querying the graph database.
    {}
    """,
        "question": """
    User question: {}
    """,
    },
}

**Schema and Graph Extraction Prompt**

In [ ]:
import json
# =============================================================================
# ADJUSTED ONTOLOGY FOR PEDIATRIC DENTISTRY & PERIODONTAL CARE
# =============================================================================

ENTITY_TYPES = [
    "CONDITION",           # e.g., Gingivitis, Periodontitis, Puberty Gingivitis
    "SYMPTOM",             # e.g., Bleeding gums, Loose teeth, Bad breath (Halitosis)
    "TREATMENT_PROCEDURE", # e.g., Scaling and Root Planing, Debridement, Antimicrobial Rinse
    "MEDICATION",          # e.g., Amoxicillin, Chlorhexidine Gluconate, Ibuprofen
    "PRE_TREATMENT_RULE",  # e.g., Fasting instructions, Medical history disclosure
    "POST_TREATMENT_CARE"  # e.g., Soft food diet, Saltwater rinses, Avoiding straws
]

GRAPH_EXTRACTION_PROMPT = """-Goal-
Given a pediatric dentistry textbook chapter focused on periodontal health, identify all medical entities of the specified types and extract clear, actionable pre-treatment, active, and post-treatment relationships.

-Steps-
1. Identify all entities. For each identified entity, extract the following information:
- entity_name: Name of the medical concept, drug, symptom, or procedure, capitalized
- entity_type: One of the following types: [{entity_types}]
- entity_description: Clear explanation of what this entity is, its clinical relevance, or specific implications for a child's health.
Format each entity exactly as: ("entity";<entity_name>;<entity_type>;<entity_description>)

2. From the entities identified in step 1, identify all pairs of (source_entity, target_entity) that are explicitly linked (e.g., a procedure treating a condition, a symptom indicating a disease, or post-treatment instructions following a specific surgery).
For each pair, extract:
- source_entity: name of the source entity, as identified in step 1
- target_entity: name of the target entity, as identified in step 1
- relationship_description: Detailed explanation of how they interact (e.g., "requires 6 hours of fasting prior to execution", "must be used for 7 days post-treatment to prevent infection")
- relationship_strength: a numeric score indicating importance or severity (0-10)
Format each relationship exactly as: ("relationship";<source_entity>;<target_entity>;<relationship_description>;<relationship_strength>)

3. Return output in English as a single list of all the entities and relationships identified. Separate each record with a single pipe character '|'. Do not add any additional text or markdown formatting blocks.

######################
-Examples-
######################
Example 1:
Entity_types: CONDITION_OR_DISEASE,DENTAL_PROCEDURE,CLINICAL_GUIDANCE,SYMPTOM_OR_SIGN
Text: For children diagnosed with localized aggressive periodontitis, deep scaling and root planing is often prescribed. Following the procedure, parents should ensure the child maintains a soft food diet for 24 hours to protect the healing gingiva and monitor for persistent bleeding.
######################
Output:
("entity";AGGRESSIVE PERIODONTITIS;CONDITION_OR_DISEASE;An uncommon, rapid type of severe gum disease affecting specific teeth in young patients)
|
("entity";SCALING AND ROOT PLANING;DENTAL_PROCEDURE;A deep cleaning procedure that removes plaque and calculus from beneath the gumline)
|
("entity";SOFT FOOD DIET;CLINICAL_GUIDANCE;Post-treatment dietary restriction to avoid irritating healing gum tissues)
|
("entity";BLEEDING;SYMPTOM_OR_SIGN;Post-operative sign indicating whether healing is processing normally or if further clinical intervention is needed)
|
("relationship";SCALING AND ROOT PLANING;AGGRESSIVE PERIODONTITIS;Is the primary active clinical treatment implemented to halt the progression of;9)
|
("relationship";SOFT FOOD DIET;SCALING AND ROOT PLANING;Must be maintained by the patient for 24 hours following the execution of;8)
|
("relationship";BLEEDING;SCALING AND ROOT PLANING;Is a critical post-operative sign that must be monitored by parents following;7)

######################
-Real Data-
######################
Entity_types: {entity_types}
Text: {input_text}
######################
Output:"""

def create_extraction_prompt(entity_types, input_text, tuple_delimiter=";"):
    return GRAPH_EXTRACTION_PROMPT.format(
        entity_types=entity_types,
        input_text=input_text,
        tuple_delimiter=tuple_delimiter,
        record_delimiter="|",
        completion_delimiter="\n\n",
    )

import re

def parse_extraction_output(output_str, tuple_delimiter=";", record_delimiter="|"):
    # Strip code fences and extra whitespace
    output_str = re.sub(r"```[a-zA-Z]*\n|\n```", "", output_str).strip()
    
    # Remove trailing completion marker (actual "\n\n")
    if output_str.endswith("\n\n"):
        output_str = output_str[:-2]
    
    entities = []
    relationships = []
    
    # Split by record_delimiter (default "|")
    records = [r.strip() for r in output_str.split(record_delimiter) if r.strip()]
    
    for rec in records:
        # Remove surrounding parentheses if present
        if rec.startswith("(") and rec.endswith(")"):
            rec = rec[1:-1]
        parts = [p.strip().strip('"\'') for p in rec.split(tuple_delimiter)]
        if not parts:
            continue
        
        if parts[0].lower() == "entity" and len(parts) >= 4:
            entities.append({
                "record_type": "entity",
                "entity_name": parts[1],
                "entity_type": parts[2],
                "entity_description": parts[3]
            })
        elif parts[0].lower() == "relationship" and len(parts) >= 5:
            try:
                strength = float(parts[4])
            except:
                strength = parts[4]
            relationships.append({
                "record_type": "relationship",
                "source_entity": parts[1],
                "target_entity": parts[2],
                "relationship_description": parts[3],
                "relationship_strength": strength
            })
    
    return entities, relationships

# ── Neo4j Cypher queries ──────────────────────────────────────────────────────
import_nodes_query = """
MERGE (b:Book {id: $book_id})
MERGE (b)-[:HAS_CHUNK]->(c:__Chunk__ {id: $chunk_id})
SET c.text = $text
WITH c
UNWIND $data AS row
MERGE (n:__Entity__ {name: row.entity_name})
// Sanitizes the medical labels by ensuring special characters or underscores don't break Cypher syntax
CALL apoc.create.addLabels(n, [row.entity_type]) YIELD node
SET n.description = coalesce(n.description, []) + [row.entity_description]
MERGE (n)<-[:MENTIONS]-(c)
"""

import_relationships_query = """
UNWIND $data AS row
MERGE (s:__Entity__ {name: row.source_entity})
MERGE (t:__Entity__ {name: row.target_entity})
CREATE (s)-[r:RELATIONSHIP {description: row.relationship_description,
                             strength: row.relationship_strength}]->(t)
"""

# ── Summarisation helpers ─────────────────────────────────────────────────────
SUMMARIZE_PROMPT = """
You are a helpful assistant responsible for generating a comprehensive summary of the data provided below.
Given one or two entities, and a list of descriptions, all related to the same entity or group of entities.
Please concatenate all of these into a single, comprehensive description.
If the provided descriptions are contradictory, please resolve the contradictions and provide a single, coherent summary.
Make sure it is written in third person, and include the entity names so we have the full context.

#######
-Data-
Entities: {entity_name}
Description List: {description_list}
#######
Output:
"""

def get_summarize_prompt(entity_name, description_list):
    return SUMMARIZE_PROMPT.format(entity_name=entity_name, description_list=description_list)

def import_entity_summary(driver, entity_information):
    driver.execute_query("""
    UNWIND $data AS row
    MATCH (e:__Entity__ {name: row.entity})
    SET e.summary = row.summary
    """, data=entity_information)
    driver.execute_query("""
    MATCH (e:__Entity__)
    WHERE size(e.description) = 1
    SET e.summary = e.description[0]
    """)

def import_rels_summary(driver, rel_summaries):
    driver.execute_query("""
    UNWIND $data AS row
    MATCH (s:__Entity__ {name: row.source}), (t:__Entity__ {name: row.target})
    MERGE (s)-[r:SUMMARIZED_RELATIONSHIP]-(t)
    SET r.summary = row.summary
    """, data=rel_summaries)
    # For single-description relationships, copy the description directly
    driver.execute_query("""
    MATCH (s:__Entity__)-[e:RELATIONSHIP]-(t:__Entity__)
    WHERE NOT (s)-[:SUMMARIZED_RELATIONSHIP]-(t)
    MERGE (s)-[r:SUMMARIZED_RELATIONSHIP]-(t)
    SET r.summary = e.description
    """)

# ── Community Detection ───────────────────────────────────────────────────────
def calculate_communities(driver):
    try:
        driver.execute_query("CALL gds.graph.drop('entity')")
    except:
        pass
    driver.execute_query("""
    MATCH (source:__Entity__)-[r:RELATIONSHIP]->(target:__Entity__)
    WITH gds.graph.project('entity', source, target, {},
         {undirectedRelationshipTypes: ['*']}) AS g
    RETURN g.graphName AS graph, g.nodeCount AS nodes, g.relationshipCount AS rels
    """)
    records, _, _ = driver.execute_query("""
    CALL gds.louvain.write("entity", {writeProperty:"louvain"})
    YIELD communityCount, communityDistribution
    RETURN communityCount, communityDistribution
    """)
    return [el.data() for el in records][0]

community_info_query = """
MATCH (e:__Entity__)
WHERE e.louvain IS NOT NULL
WITH e.louvain AS louvain, collect(e) AS nodes
WHERE size(nodes) > 1
CALL apoc.path.subgraphAll(nodes[0], {whitelistNodes: nodes})
YIELD relationships
RETURN louvain AS communityId,
       [n in nodes | {
           id: n.name, 
           description: n.summary,
           type: coalesce([el in labels(n) WHERE el <> '__Entity__'][0], 'MEDICAL_CONCEPT')
       }] AS nodes,
       [r in relationships | {
           start: startNode(r).name, 
           type: type(r),
           end: endNode(r).name, 
           description: r.description
       }] AS rels
"""

COMMUNITY_REPORT_PROMPT = """
You are an AI assistant that helps a human analyst to perform general information discovery.

# Goal
Write a comprehensive report of a community, given a list of entities that belong to the community
as well as their relationships.

# Report Structure
The report should include:
- TITLE: short but specific community name
- SUMMARY: executive summary of the community
- IMPACT SEVERITY RATING: float score 0-10
- RATING EXPLANATION: single sentence explanation
- DETAILED FINDINGS: list of 5-10 key insights

Return output as a well-formed JSON string with this format:
{{
    "title": "<report_title>",
    "summary": "<executive_summary>",
    "rating": <impact_severity_rating>,
    "rating_explanation": "<rating_explanation>",
    "findings": [
        {{"summary": "<insight_1_summary>", "explanation": "<insight_1_explanation>"}}
    ]
}}

# Input Data
{input_text}

Output:"""

def get_summarize_community_prompt(nodes, relationships):
    input_text = f"Entities\n    {nodes}\n\n    Relationships\n    {relationships}\n    "
    return COMMUNITY_REPORT_PROMPT.format(input_text=input_text)

def extract_json(input_str: str):
    return input_str.removeprefix("```json").removesuffix("```").strip()

import_community_query = """
UNWIND $data AS row
MERGE (c:__Community__ {communityId: row.communityId})
SET c.title = row.community.title,
    c.summary = row.community.summary,
    c.rating = row.community.rating,
    c.rating_explanation = row.community.rating_explanation
WITH c, row
UNWIND row.nodes AS node
MERGE (n:__Entity__ {name: node})
MERGE (n)-[:IN_COMMUNITY]->(c)
"""

print("Extra functions initialized.")


**DATA LOAD**

In [ ]:
import json
import os
import tkinter as tk
from tkinter import filedialog
from typing import List

# Setup tkinter file dialog
root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)

print("Opening dialog to select your JSON knowledge base file...")
json_path = filedialog.askopenfilename(
    title="Select the Extracted Chapters JSON File",
    filetypes=[("JSON Files", "*.json")]
)
root.destroy()

if json_path:
    print(f"[✓] Selected File: {json_path}")
else:
    print("[!] No file selected.")

In [ ]:
# 1. Load the structured JSON data
with open(json_path, 'r', encoding='utf-8') as f:
    knowledge_base = json.load(f)

# 2. Extract text blocks into the 'books' list to preserve your original workflow
books = [source_data["content"] for label, source_data in knowledge_base.items()]
book_labels = [label for label in knowledge_base.keys()] 

print("Calculating token stats via local Ollama (qwen2.5:7b)... Please wait.")

# 3. Print book token stats (Fixed: safely returns integers now)
token_count = [num_tokens_from_string(el, model_name="qwen2.5:7b") for el in books]

print(f"""\nThere are {len(token_count)} chapters/sources with token sizes:
- avg {sum(token_count) / len(token_count):.1f}
- min {min(token_count)}
- max {max(token_count)}
""")

# 4. Create chunked books (Your original character-based window splitter)
chunked_books = [chunk_text(book, 2000, 400) for book in books]

print("\n--- Chunking Results Summary ---")
for i, label in enumerate(book_labels):
    print(f"'{label}' has been split into {len(chunked_books[i])} chunks.")

**Chunking Result Validation**

In [ ]:
# --- VALIDATION CELL ---
# Inspecting the first chapter (index 0) and its first 2 chunks

chapter_idx = 0  # Change this to inspect different chapters (0, 1, 2...)
chunk_idx = 0    # Change this to inspect subsequent chunks

if len(chunked_books) > chapter_idx and len(chunked_books[chapter_idx]) > chunk_idx:
    sample_chunk = chunked_books[chapter_idx][chunk_idx]
    print(f"--- Validating: {book_labels[chapter_idx]} | Chunk {chunk_idx + 1} ---")
    print(f"Total Chunks in this chapter: {len(chunked_books[chapter_idx])}")
    print(f"Chunk Length (chars): {len(sample_chunk)}")
    print("\n--- Content Preview ---")
    print(sample_chunk[:500] + "...") # Prints first 500 characters
else:
    print("Chunk index out of range. Check your indices.")

In [ ]:
# ===== ENHANCED EXTRACTION FUNCTION =====
def extract_entities(text, entity_types=None):
    if entity_types is None:
        entity_types = ENTITY_TYPES

    prompt = GRAPH_EXTRACTION_PROMPT.format(
        entity_types=",".join(entity_types),
        input_text=text
    )
    
    print(f"Sending prompt of length {len(prompt)} to model...")
    
    output_str = chat(
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        model=LOCAL_MODEL_NAME
    )
    
    print(f"Received output of length: {len(output_str) if output_str else 0}")
    if not output_str:
        print("chat() returned empty string or None. Cannot proceed.")
        return [], []
    
    # --- Enhanced Cleaning ---
    # 1. Remove markdown code blocks robustly
    output_str = re.sub(r"```[a-zA-Z]*\n", "", output_str)
    output_str = re.sub(r"\n```", "", output_str)
    
    # 2. Remove trailing completion marker
    if output_str.endswith("\n\n"):
        output_str = output_str[:-2]
        
    # 3. Trim leading/trailing whitespace
    output_str = output_str.strip()
    
    # 4. Additional logging to see the cleaned output
    print(" First 500 chars of CLEANED response:\n", output_str[:500])
    
    nodes, relationships = parse_extraction_output(
        output_str,
        record_delimiter="|",
        tuple_delimiter=";"
    )
    print(f"Parsed {len(nodes)} entities and {len(relationships)} relationships.")
    return nodes, relationships

In [ ]:
import time
from tqdm import tqdm

# ===== CONFIGURATION =====
EXTRACTION_MODEL = LOCAL_EMBED_MODEL     # Replaced LOCAL_MODEL_NAME with your explicit Qwen model string
DELAY_BETWEEN_CHUNKS = 3                # seconds delay to give local hardware/Ollama a breathing pause
number_of_books = 1                     # Process only the first chapter ('source 1') for testing; change to len(chunked_books) later
# =========================

for book_i in range(number_of_books):
    # 1. CHANGE: Replaced pdf_chunks with chunks from the active chapter
    chapter_chunks = chunked_books[book_i]
    
    # Safely grab the label we generated in the previous step (e.g., 'source 1')
    current_source_label = book_labels[book_i] if 'book_labels' in locals() else f"source_{book_i + 1}"
    
    print(f"\n📖 Processing {current_source_label.upper()} – {len(chapter_chunks)} chunks")
    print(f" Configured Model: {EXTRACTION_MODEL} | Delay: {DELAY_BETWEEN_CHUNKS}s between chunks\n")

    consecutive_errors = 0

    # 2. CHANGE: Corrected nesting indentation for the ingestion loop
    for chunk_i, chunk in enumerate(tqdm(chapter_chunks, desc=f"Ingesting {current_source_label}", unit="chunk")):
        try:
            # Directly extract using your local Qwen model parameters
            nodes, relationships = extract_entities(chunk, entity_types=ENTITY_TYPES)
            
            if not nodes and not relationships:
                time.sleep(DELAY_BETWEEN_CHUNKS)
                continue

            if nodes:
                neo4j_driver.execute_query(
                    import_nodes_query,
                    data=nodes,
                    # 3. CHANGE: Dynamically tag the book_id with the precise chapter label
                    book_id=f"Dentistry_Child_Adolescent_8th_{current_source_label}",
                    text=chunk,
                    chunk_id=chunk_i,
                )
            if relationships:
                neo4j_driver.execute_query(
                    import_relationships_query,
                    data=relationships,
                )
            
            # Reset error count on a successful operation
            consecutive_errors = 0
            
            # 4. CHANGE: Implemented your configured delay for local inference stabilization
            time.sleep(DELAY_BETWEEN_CHUNKS)

        except Exception as e:
            tqdm.write(f"  ✗ Processing chunk {chunk_i+1} failed: {e}")
            consecutive_errors += 1
            
            # Smart fallback: if Ollama hits VRAM caps or crashes, take a longer recovery nap
            if consecutive_errors >= 3:
                tqdm.write(" 3 consecutive errors encountered. Sleeping 15s to allow local Ollama to flush context...")
                time.sleep(15)
            else:
                time.sleep(2)
            continue

print("\n✅ Ingestion complete.")

**INGESTION RESULT VERIFICATION**

In [ ]:
# Verify entity and relationship counts
data, _, _ = neo4j_driver.execute_query("""
    MATCH (:`__Entity__`)
    RETURN 'entity' AS type, count(*) AS count
    UNION
    MATCH ()-[:RELATIONSHIP]->()
    RETURN 'relationship' AS type, count(*) AS count
""")
print([el.data() for el in data])

**ENTITIY SUMARIZATION**

In [ ]:
candidates_to_summarize, _, _ = neo4j_driver.execute_query("""
    MATCH (e:__Entity__) WHERE size(e.description) > 1
    RETURN e.name AS entity_name, e.description AS description_list
""")

summaries = []
for candidate in tqdm(candidates_to_summarize, desc="Summarising entities"):
    messages = [{
        "role": "user",
        "content": get_summarize_prompt(candidate["entity_name"], candidate["description_list"]),
    }]
    summary = chat(messages)
    summaries.append({"entity": candidate["entity_name"], "summary": summary})

import_entity_summary(neo4j_driver, summaries)
print(f"Summarised {len(summaries)} entities.")

In [ ]:
import time
from tqdm import tqdm

# Ensure your model variable is set
EXTRACTION_MODEL = LOCAL_MODEL_NAME

rels_to_summarize, _, _ = neo4j_driver.execute_query("""
    MATCH (s:__Entity__)-[r:RELATIONSHIP]-(t:__Entity__)
    WHERE id(s) < id(t)
    WITH s.name AS source, t.name AS target,
         collect(r.description) AS description_list,
         count(*) AS cnt
    WHERE cnt > 1
    RETURN source, target, description_list
""")

print(f"Found {len(rels_to_summarize)} relationship pairs to summarize.")

rel_summaries = []
for candidate in tqdm(rels_to_summarize, desc="Summarising relationships"):
    entity_name = f"{candidate['source']} relationship to {candidate['target']}"
    
    # PRO TIP: Deduplicate descriptions to save massive amounts of tokens!
    unique_descriptions = list(set(candidate["description_list"]))
    
    messages = [{
        "role": "user",
        "content": get_summarize_prompt(entity_name, unique_descriptions),
    }]
    
    # Explicitly pass the model
    summary = chat(messages, model=EXTRACTION_MODEL)
    
    rel_summaries.append({
        "source": candidate["source"], 
        "target": candidate["target"], 
        "summary": summary
    })
    
    # THE FIX: Throttle the loop so you don't instantly blow through the RPM limits of all your keys
    time.sleep(2) 

import_rels_summary(neo4j_driver, rel_summaries)
print(f"Created SUMMARIZED_RELATIONSHIP edges for {len(rel_summaries)} relationship pairs.")


**SUMMARIZATION RESULT VERIFICATION**

In [ ]:
# Verify SUMMARIZED_RELATIONSHIP was created
data, _, _ = neo4j_driver.execute_query("""
    MATCH ()-[r:SUMMARIZED_RELATIONSHIP]-()
    RETURN count(r) AS summarized_rel_count
""")
print([el.data() for el in data])

**COMMUNITY DISTRIBUTION CHECK VIA NetworkX**

In [ ]:
import json
import networkx as nx

from networkx.algorithms.community import louvain_communities
from tqdm import tqdm

# ===== NETWORKX FALLBACK FOR NEO4J AURA FREE TIER =====
def calculate_communities_local(driver):
    print("Fetching graph topology from Neo4j Aura Free...")
    query = """
    MATCH (s:__Entity__)-[:RELATIONSHIP]->(t:__Entity__)
    RETURN s.name AS source, t.name AS target
    """
    records, _, _ = driver.execute_query(query)
    
    # 1. Build NetworkX undirected graph
    G = nx.Graph()
    for record in records:
        G.add_edge(record["source"], record["target"])
        
    # Ensure any isolated entities are captured as well
    all_nodes_query = "MATCH (e:__Entity__) RETURN e.name AS name"
    node_records, _, _ = driver.execute_query(all_nodes_query)
    for r in node_records:
        G.add_node(r["name"])
        
    print("Running Louvain Community Detection locally via NetworkX...")
    communities = louvain_communities(G, seed=42)
    # Sort communities from largest to smallest to match the professor's distribution order
    communities = sorted(communities, key=len, reverse=True)
    
    print("Synchronizing community properties back to your Neo4j instance...")
    for community_id, node_set in enumerate(communities):
        driver.execute_query("""
        UNWIND $nodes AS node_name
        MATCH (e:__Entity__ {name: node_name})
        SET e.louvain = $community_id
        """, nodes=list(node_set), community_id=community_id)
        
    # 4. Format the distribution exactly like the professor's expected dictionary format
    custom_distribution = {i: len(c) for i, c in enumerate(communities)}
    
    return {
        "communityCount": len(communities),
        "communityDistribution": custom_distribution
    }
# ======================================================

community_distribution = calculate_communities_local(neo4j_driver)
print(f"There are {community_distribution['communityCount']} communities with distribution: {community_distribution['communityDistribution']}")

**COMMUNITY SUMMARIZATION**

In [ ]:
community_info, _, _ = neo4j_driver.execute_query(community_info_query)
communities = []
for community in tqdm(community_info, desc="Summarizing communities"):
    messages = [
        {
            "role": "user",
            "content": get_summarize_community_prompt(
                community["nodes"], community["rels"]
            ),
        },
    ]
    summary = chat(messages, model=EXTRACTION_MODEL)
    try:
        communities.append(
            {
                "community": json.loads(extract_json(summary)),
                "communityId": community["communityId"],
                "nodes": [el["id"] for el in community["nodes"]],
            }
        )
    except json.JSONDecodeError:
        tqdm.write(f" Skipping malformed community JSON for ID {community['communityId']}")

neo4j_driver.execute_query(import_community_query, data=communities)

**GRAPH EMBEDDEDING AND CONTRUCTION VECTOR INDEXING**

In [ ]:
entities, _, _ = neo4j_driver.execute_query("MATCH (e:__Entity__) RETURN e.summary AS summary, e.name AS name")
data = [{"name": el["name"], "embedding": embed(el["summary"] if el["summary"] else el["name"])} for el in entities]

neo4j_driver.execute_query(
    "UNWIND $data AS row MATCH (e:__Entity__ {name: row.name}) CALL db.create.setNodeVectorProperty(e, 'embedding', row.embedding)",
    data=data,
)
neo4j_driver.execute_query(
    "CREATE VECTOR INDEX entities IF NOT EXISTS FOR (n:__Entity__) ON (n.embedding)"
)
print("Graph analytics pipeline complete. Vector index is ready for retrieval.")